In [ ]:
import torch
import torch.nn as nn

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"

#Model for Instruction Fine Tuning

In [ ]:
GPT_CONFIG_355M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 1024,
    "n_heads": 16,
    "n_layers": 24,
    "drop_rate": 0.0,
    "qkv_bias": True
}

In [ ]:
class MultiheadAttention(nn.Module):
  def __init__(self,d_in,d_out,context_length,n_heads,dropout,qkv_bias):
    super().__init__()
    assert (d_out % n_heads == 0), "d_out must be div by n_heads"

    self.head_dim=d_out // n_heads
    self.n_heads=n_heads
    self.d_out=d_out

    self.w_q=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.w_k=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.w_v=nn.Linear(d_in,d_out,bias=qkv_bias)

    self.out_proj=nn.Linear(d_out,d_out)
    self.dropout=nn.Dropout(dropout)

    self.register_buffer(
        "mask",
        torch.triu(
            torch.ones(context_length,context_length),
            diagonal=1
        )
    )

  def forward(self,x):
    b,cn,emb=x.shape

    keys=self.w_k(x)
    queries=self.w_q(x)
    values=self.w_v(x)

    keys=keys.view(b,cn,self.n_heads,self.head_dim)
    queries=queries.view(b,cn,self.n_heads,self.head_dim)
    values=values.view(b,cn,self.n_heads,self.head_dim)

    keys=keys.transpose(1,2)
    queries=queries.transpose(1,2)
    values=values.transpose(1,2)

    attn_scores=queries @ keys.transpose(2,3)

    attn_scores.masked_fill_(
        self.mask[:cn,:cn].bool(),
        -torch.inf
    )

    attn_weights=torch.softmax(
        attn_scores/(self.head_dim ** 0.5),
        dim=-1
    )

    attn_weights=self.dropout(attn_weights)

    context_vec=(attn_weights @ values).transpose(1,2)

    context_vec=context_vec.contiguous().view(
        b,
        cn,
        self.d_out
    )

    context_vec=self.out_proj(context_vec)

    return context_vec

In [ ]:
class Layer_norm(nn.Module):
  def __init__(self,dim):
    super().__init__()
    self.eps=1e-5
    self.scale=nn.Parameter(torch.ones(dim))
    self.shift=nn.Parameter(torch.zeros(dim))
  def forward(self,x):
    mean=x.mean(dim=-1,keepdim=True)
    var=x.var(dim=-1,keepdim=True,unbiased=False)
    norm=(x-mean)/torch.sqrt(var+self.eps)
    return self.scale*norm+self.shift


In [ ]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()
    self.gelu = nn.GELU(approximate='tanh')
  def forward(self,x):
    return self.gelu(x)

In [ ]:
class FeedForward(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.layer=nn.Sequential(
        nn.Linear(cfg["emb_dim"],cfg["emb_dim"]*4),
        GELU(),
        nn.Linear(cfg["emb_dim"]*4,cfg["emb_dim"])
    )
  def forward(self,x):
    return self.layer(x)

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.Mha=MultiheadAttention(
        d_in=cfg["emb_dim"],
        d_out=cfg["emb_dim"],
        context_length=cfg["context_length"],
        n_heads=cfg["n_heads"],
        dropout=cfg["drop_rate"],
        qkv_bias=cfg["qkv_bias"]
    )
    self.ffn=FeedForward(cfg)
    self.norm1 = Layer_norm(cfg["emb_dim"])
    self.norm2 = Layer_norm(cfg["emb_dim"])
    self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
  def forward(self,x):
    shortcut=x
    x=self.norm1(x)
    x=self.Mha(x)
    x=self.drop_shortcut(x)
    x=x+shortcut
    shortcut=x
    x=self.norm2(x)
    x=self.ffn(x)
    x=self.drop_shortcut(x)
    x=x+shortcut
    return x


In [ ]:
class GPTModel(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.tok_emb=nn.Embedding(config["vocab_size"],config["emb_dim"])
    self.pos_emb=nn.Embedding(config["context_length"],config["emb_dim"])
    self.Dropout=nn.Dropout(config["drop_rate"])
    self.trf_blocks=nn.Sequential(*[TransformerBlock(config) for _ in range(config["n_layers"])])
    self.final_norm=Layer_norm(config["emb_dim"])
    self.out_head=nn.Linear(config["emb_dim"],config["vocab_size"],bias=False)
  def forward(self,ips):
    batch_shape,seq_len=ips.shape
    tok_embs=self.tok_emb(ips)
    pos_embs=self.pos_emb(torch.arange(seq_len,device=ips.device))
    x=tok_embs+pos_embs
    x=self.Dropout(x)
    x=self.trf_blocks(x)
    x=self.final_norm(x)
    logits=self.out_head(x)
    return logits

In [ ]:
model=GPTModel(GPT_CONFIG_355M)

#Loading OpenAI weights

In [ ]:
model.load_state_dict(torch.load("gpt3_weights.pt",map_location=torch.device('cpu')))

In [ ]:
import os

file_size = os.path.getsize("gpt3_weights.pt") / (1024 * 1024)
print(f"File size: {file_size:.2f} MB")

File size: 4.00 MB


In [ ]:
from gpt3_download import download_and_load_gpt2
settings,params=download_and_load_gpt2("355M","gpt2_weights")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 227kiB/s]
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
encoder.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 3.46MiB/s]
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verificat

In [ ]:
print(type(params))
print(params.keys())
print(params["blocks"][0].keys())
print(params["blocks"][0]["mlp"].keys())

<class 'dict'>
dict_keys(['blocks', 'b', 'g', 'wpe', 'wte'])
dict_keys(['attn', 'ln_1', 'ln_2', 'mlp'])
dict_keys(['c_fc', 'c_proj'])


In [ ]:
def assign(l,r):
  return torch.nn.Parameter(torch.tensor(r))

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params):

    gpt.pos_emb.weight = assign(
        gpt.pos_emb.weight,
        params["wpe"]
    )

    gpt.tok_emb.weight = assign(
        gpt.tok_emb.weight,
        params["wte"]
    )

    for b in range(len(params["blocks"])):

        # QKV weights
        q_w, k_w, v_w = np.split(
            params["blocks"][b]["attn"]["c_attn"]["w"],
            3,
            axis=-1
        )

        gpt.trf_blocks[b].Mha.w_q.weight = assign(
            gpt.trf_blocks[b].Mha.w_q.weight,
            q_w.T
        )

        gpt.trf_blocks[b].Mha.w_k.weight = assign(
            gpt.trf_blocks[b].Mha.w_k.weight,
            k_w.T
        )

        gpt.trf_blocks[b].Mha.w_v.weight = assign(
            gpt.trf_blocks[b].Mha.w_v.weight,
            v_w.T
        )

        # QKV biases
        q_b, k_b, v_b = np.split(
            params["blocks"][b]["attn"]["c_attn"]["b"],
            3,
            axis=-1
        )

        gpt.trf_blocks[b].Mha.w_q.bias = assign(
            gpt.trf_blocks[b].Mha.w_q.bias,
            q_b
        )

        gpt.trf_blocks[b].Mha.w_k.bias = assign(
            gpt.trf_blocks[b].Mha.w_k.bias,
            k_b
        )

        gpt.trf_blocks[b].Mha.w_v.bias = assign(
            gpt.trf_blocks[b].Mha.w_v.bias,
            v_b
        )

        # Attention output projection
        gpt.trf_blocks[b].Mha.out_proj.weight = assign(
            gpt.trf_blocks[b].Mha.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T
        )

        gpt.trf_blocks[b].Mha.out_proj.bias = assign(
            gpt.trf_blocks[b].Mha.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"]
        )

        # Feed Forward first linear
        gpt.trf_blocks[b].ffn.layer[0].weight = assign(
            gpt.trf_blocks[b].ffn.layer[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T
        )

        gpt.trf_blocks[b].ffn.layer[0].bias = assign(
            gpt.trf_blocks[b].ffn.layer[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"]
        )

        # Feed Forward second linear
        gpt.trf_blocks[b].ffn.layer[2].weight = assign(
            gpt.trf_blocks[b].ffn.layer[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T
        )

        gpt.trf_blocks[b].ffn.layer[2].bias = assign(
            gpt.trf_blocks[b].ffn.layer[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"]
        )

        # LayerNorm 1
        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"]
        )

        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"]
        )

        # LayerNorm 2
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"]
        )

        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"]
        )

    # Final LayerNorm
    gpt.final_norm.scale = assign(
        gpt.final_norm.scale,
        params["g"]
    )

    gpt.final_norm.shift = assign(
        gpt.final_norm.shift,
        params["b"]
    )

    # Weight tying (GPT-2)
    gpt.out_head.weight = assign(
        gpt.out_head.weight,
        params["wte"]
    )

In [ ]:
load_weights_into_gpt(model,params)

In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken
tokenizer=tiktoken.get_encoding("gpt2")
text="Every effort moves you"
model.to(device)
logits=model(torch.tensor(tokenizer.encode(text)).unsqueeze(0).to(device))
print(tokenizer.decode([torch.argmax(logits[0,-1])]))

 forward


In [ ]:
torch.save(model.state_dict(),"gpt3_weights.pt")

# importing Data and DataLoaders

In [ ]:
import json
with open("instruction-data.json","r",encoding="utf-8") as file:
  data=json.load(file)
print(len(data))
print(type(data[0]))

1100
<class 'dict'>


In [ ]:
train_ratio=0.9
split_idx=int(len(data)*train_ratio)
train_data=data[:split_idx]
test_data=data[split_idx:]
print(len(train_data))

990


In [ ]:
def format_input(entry, is_training=True):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry.get("input") else ""
    response_header = "\n\n### Response:\n"
    full_prompt = instruction_text + input_text + response_header
    if is_training:
        return full_prompt + entry["output"]
    else:
        return full_prompt

In [ ]:
train_data = [format_input(data) for data in train_data]
test_data=[format_input(data) for data in test_data]

In [ ]:
import torch
from torch.utils.data import Dataset,DataLoader

class CreateDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.encoded = []
        for text in data:
          input_ids = tokenizer.encode(text)
          self.encoded.append(input_ids)
    def __len__(self):
        return len(self.encoded)

    def __getitem__(self, index):
        return self.encoded[index]

In [ ]:
train_data=CreateDataset(train_data,tokenizer)
test_data=CreateDataset(test_data,tokenizer)

In [ ]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cuda"
):
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst, targets_lst = [], []
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))

        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = (targets == pad_token_id)
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [ ]:
train_loader=DataLoader(
    dataset=train_data,
    batch_size=8,
    num_workers=0,
    shuffle=True,
    drop_last=True,
    collate_fn=custom_collate_fn
)
test_loader=DataLoader(
    dataset=test_data,
    batch_size=8,
    num_workers=0,
    shuffle=True,
    drop_last=True,
    collate_fn=custom_collate_fn
)

#Loss function,Accuracy Loader

In [63]:
def calc_loss_batch(inputs, targets, model, device):
    inputs, targets = inputs.to(device), targets.to(device)

    logits = model(inputs)
    loss = torch.nn.functional.cross_entropy(
        logits.view(-1, logits.size(-1)),
        targets.view(-1),
        ignore_index=-100
    )

    return loss

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [60]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)
            with torch.no_grad():
                logits = model(input_batch)
            predicted_labels = torch.argmax(logits, dim=-1)
            valid_mask = (target_batch != -100)

            correct_tokens = (predicted_labels == target_batch) & valid_mask
            correct_predictions += correct_tokens.sum().item()
            num_examples += valid_mask.sum().item() # Count only actual response tokens
        else:
            break

    # Avoid DivisionByZero if a batch somehow has zero valid tokens
    if num_examples == 0:
        return 0.0

    return correct_predictions / num_examples

#Training Loop

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [ ]:
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                            eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            examples_seen += input_batch.shape[0]
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

In [61]:
import time

start_time = time.time()

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, test_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000000): Train loss 0.292, Val loss 0.743
Ep 1 (Step 000050): Train loss 0.273, Val loss 0.816
Ep 1 (Step 000100): Train loss 0.249, Val loss 0.845
Training accuracy: 93.70% | Validation accuracy: 87.04%
Ep 2 (Step 000150): Train loss 0.208, Val loss 0.730
Ep 2 (Step 000200): Train loss 0.192, Val loss 0.746
Training accuracy: 95.25% | Validation accuracy: 85.96%
Ep 3 (Step 000250): Train loss 0.187, Val loss 0.823
Ep 3 (Step 000300): Train loss 0.167, Val loss 0.731
Ep 3 (Step 000350): Train loss 0.164, Val loss 0.801
Training accuracy: 96.01% | Validation accuracy: 86.44%
Ep 4 (Step 000400): Train loss 0.164, Val loss 0.783
Ep 4 (Step 000450): Train loss 0.172, Val loss 0.901
Training accuracy: 95.30% | Validation accuracy: 87.17%
Ep 5 (Step 000500): Train loss 0.148, Val loss 0.884
Ep 5 (Step 000550): Train loss 0.159, Val loss 0.848
Ep 5 (Step 000600): Train loss 0.162, Val loss 0.868
Training accuracy: 95.52% | Validation accuracy: 85.81%
Training completed in 5.46 minu

#Evaluation

In [68]:
text = {"instruction": "what is the melting point of iron", "input": "", "output": ""}
formatted_text = format_input(text, False)
input_ids = torch.tensor(tokenizer.encode(formatted_text)).unsqueeze(0).to(device)

for _ in range(30):
    with torch.no_grad():
        logits = model(input_ids)
    next_token_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

    input_ids = torch.cat((input_ids, next_token_id), dim=-1)

    decoded_token = tokenizer.decode(next_token_id[0].tolist())
    print(decoded_token, end="", flush=True)

    if next_token_id.item() == 50256:
        break

The melting point of iron is 1538 degrees Celsius.<|endoftext|>